# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema and accessible online.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings

warnings.filterwarnings('ignore')
# Set display option for Pandas
pd.set_option('display.max_columns', None)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}\n---\n{metadata.description}\n")

## 2. Data Overview
Let's list all record sets, their IDs, and their fields as defined by their `@id` in the Croissant schema.

In [ ]:
from pprint import pprint

record_sets = dataset.record_sets
record_set_ids = list(record_sets)
print("Available Record Sets and their fields by @id:")
for rs_id, rs in record_sets.items():
    print(f"\nRecordSet @id: {rs_id}")
    print(f"Name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print("Fields:")
    for field_id, field in rs.fields.items() if hasattr(rs, 'fields') else []:
        print(f"  Field @id: {field_id} (name: {field.name})")

## 3. Data Extraction
Extract data from each main record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
# For demonstration, pick the main survey record set (commonly it's '@id': 'survey', but let's detect)
all_dataframes = {}
for rs_id in record_sets:
    print(f'Loading data from RecordSet: {rs_id}')
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        print(f'DataFrame shape: {df.shape}')
        all_dataframes[rs_id] = df
    except Exception as e:
        print(f'Could not load records for {rs_id}:', e)
if all_dataframes:
    # Pick first record set for demonstration
    main_record_set_id = list(all_dataframes.keys())[0]
    main_df = all_dataframes[main_record_set_id]
    print(f'\nColumns in main record set ({main_record_set_id}):')
    print(main_df.columns.tolist())
    main_df.head()
else:
    print('No record sets with extractable records found.')

## 4. Exploratory Data Analysis (EDA)
We will demonstrate common operations using the primary survey DataFrame.
1. Filtering records by a numeric field (e.g., 'gad7_total_score' > 10)
2. Normalizing the 'gad7_total_score' column
3. Grouping by demographic field (e.g., 'gender') and computing mean scores

**Note:** All field names and column selectors are referenced by their `@id`, as per Croissant specification.

In [ ]:
# Select a numeric field (`@id` assumed from standard instruments)
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    if 'gad7_total_score' in col:
        numeric_field_id = col
    if 'gender' in col:
        group_field_id = col
if numeric_field_id is None:
    # Fallback to any integer/float field
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
if group_field_id is None:
    for col in main_df.columns:
        if 'sex' in col or 'gender' in col:
            group_field_id = col
            break

print(f"Numeric field selected (by @id): {numeric_field_id}")
print(f"Grouping field selected (by @id): {group_field_id}\n")

# Filtering
threshold = 10
if numeric_field_id:
    filtered_df = main_df[main_df[numeric_field_id].astype('float').fillna(0) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (
        filtered_df[numeric_field_id].astype('float') - filtered_df[numeric_field_id].astype('float').mean()
    ) / filtered_df[numeric_field_id].astype('float').std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by '{group_field_id}':")
        display(grouped_df)
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Let's visualize the distribution of the selected score, and group-wise mean values using matplotlib (or seaborn if desired).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna().astype(float), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (GAD-7 Total Score Example)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to access and explore the FAIR² Mental Health Survey Dataset from Kilifi, Kenya using the `mlcroissant` library.

**Key steps included:**
- Loading metadata and listing record sets by their `@id`
- Extracting data for analysis, with all fields referenced by their Croissant `@id`
- Demonstrating basic EDA: filtering, normalization, and group-by
- Visualizing score distributions and demographic differences

The dataset provides valuable insight into mental health patterns, survey-based instrument scores, and demographics in Kilifi. For further research, explore additional fields and conduct advanced analyses using field `@id`s.